# SAR Oil Spill Detection — Training Notebook

Trains U-Net directly on raw `.tif` files from Google Drive.
No patch extraction needed — dataset.py crops on-the-fly.

## Before running:
1. Run `colab_preprocessing.ipynb` once to generate `train_stats.json` and `splits.json`
2. Go to **Runtime → Change runtime type → T4 GPU**
3. Run all cells top to bottom

---
## Cell 1 — Verify GPU

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Go to Runtime -> Change runtime type -> T4 GPU')

---
## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
def gb(x): return x / (1024**3)
_, _, free = shutil.disk_usage('/content')
print(f'Colab local disk free: {gb(free):.1f} GB')

---
## Cell 3 — Locate Dataset and Stats

Finds the dataset folder and the stats files saved by the preprocessing notebook.
The dataset `.tif` files stay on Drive — we read them directly during training.

In [ ]:
import os, json

# ── Find dataset folder ─────────────────────────────────────────────────────
DATASET_PATH = '/content/drive/MyDrive/Geo_Spill_Data'

# ── Find stats files ────────────────────────────────────────────────────────
RESULTS_DIR = '/content/drive/MyDrive/Geo_Spill_results'
STATS_DRIVE  = f'{RESULTS_DIR}/train_stats.json'
SPLITS_DRIVE = f'{RESULTS_DIR}/splits.json'

if not os.path.exists(STATS_DRIVE):
    raise FileNotFoundError(
        f'Stats not found at {STATS_DRIVE}\n'
        'Run colab_preprocessing.ipynb first.'
    )

with open(STATS_DRIVE) as f:  stats  = json.load(f)
with open(SPLITS_DRIVE) as f: splits = json.load(f)

print(f'Dataset         : {DATASET_PATH}')
print(f'Mean (VV, VH)   : {stats["mean"]}')
print(f'Std  (VV, VH)   : {stats["std"]}')
print(f'Split           : {len(splits["train"])} train | {len(splits["val"])} val | {len(splits["test"])} test')

# ── Rewrite splits.json paths to point at Drive ─────────────────────────────
# The splits.json was generated with local paths — we remap them to Drive paths
def remap_path(p, dataset_path):
    fname = os.path.basename(p)
    if '_mask' in fname or fname in os.listdir(f'{dataset_path}/masks'):
        return f'{dataset_path}/masks/{fname}'
    return f'{dataset_path}/images/{fname}'

splits['train'] = [f'{DATASET_PATH}/images/{os.path.basename(p)}' for p in splits['train']]
splits['val']   = [f'{DATASET_PATH}/images/{os.path.basename(p)}' for p in splits['val']]
splits['test']  = [f'{DATASET_PATH}/images/{os.path.basename(p)}' for p in splits['test']]
splits['masks'] = {
    f'{DATASET_PATH}/images/{os.path.basename(k)}': f'{DATASET_PATH}/masks/{os.path.basename(v)}'
    for k, v in splits['masks'].items()
}

# Save remapped splits locally for train.py to use
os.makedirs('data', exist_ok=True)
with open('data/splits.json', 'w') as f:      json.dump(splits, f)
with open('data/train_stats.json', 'w') as f: json.dump(stats, f)
print('\nSplits remapped to Drive paths and saved locally.')

---
## Cell 4 — Clone GitHub Repository

In [ ]:
import os

GITHUB_URL = 'https://github.com/TigranBoyakhchyan/GeoSpill-AI'
REPO_NAME  = 'GeoSpill-AI'

if os.path.exists(f'/content/{REPO_NAME}'):
    %cd /content/{REPO_NAME}
    !git pull origin main
else:
    !git clone {GITHUB_URL}
    %cd /content/{REPO_NAME}

# Symlink the data/ folder into the repo
if not os.path.exists('data'):
    os.symlink('/content/data', 'data')

print(f'Working directory: {os.getcwd()}')

---
## Cell 5 — Install Dependencies

In [ ]:
!pip install -q segmentation-models-pytorch albumentations rasterio
import segmentation_models_pytorch as smp
import albumentations as A
print(f'smp: {smp.__version__} | albumentations: {A.__version__}')

---
## Cell 6 — Train

- `--crops_per_image 10` → 1200 images × 10 crops = 12,000 samples per epoch
- Images are read from Drive on-the-fly — no disk space issues
- Expected time: **20-40 minutes** on T4 GPU for 50 epochs

In [ ]:
!python src/train.py \
    --epochs 50 \
    --batch_size 16 \
    --lr 1e-4 \
    --model_type smp \
    --crops_per_image 10

---
## Cell 7 — Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('checkpoints/training_log.csv')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Curves', fontsize=14, fontweight='bold')

for ax, metric, title in zip(axes, ['loss', 'iou'], ['Loss', 'IoU']):
    ax.plot(log['epoch'], log[f'train_{metric}'], label=f'Train {title}', color='steelblue')
    ax.plot(log['epoch'], log[f'val_{metric}'],   label=f'Val {title}',   color='orange')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

best = log.loc[log['val_iou'].idxmax()]
print(f'Best epoch   : {int(best["epoch"])}')
print(f'Best val IoU : {best["val_iou"]:.4f}')
print(f'Best val Dice: {best["val_dice"]:.4f}')

---
## Cell 8 — Save Results to Drive
**Always run before closing the session.**

In [ ]:
import shutil, os

SAVE_DIR = '/content/drive/MyDrive/oil_spill_results'
os.makedirs(SAVE_DIR, exist_ok=True)

for src, dst in {
    'checkpoints/best_model.pth':      f'{SAVE_DIR}/best_model.pth',
    'checkpoints/training_log.csv':    f'{SAVE_DIR}/training_log.csv',
    'checkpoints/training_curves.png': f'{SAVE_DIR}/training_curves.png',
}.items():
    shutil.copy(src, dst)
    print(f'Saved: {src} -> Drive')

---
## Cell 9 — Commit Results to GitHub
**Replace email and name.**

In [ ]:
import pandas as pd
log  = pd.read_csv('checkpoints/training_log.csv')
best = log.loc[log['val_iou'].idxmax()]

!git config user.email "you@example.com"
!git config user.name  "Your Name"
!git add checkpoints/training_log.csv checkpoints/training_curves.png data/train_stats.json

msg = f'Colab training: val IoU={best["val_iou"]:.4f}, Dice={best["val_dice"]:.4f} ({len(log)} epochs)'
!git commit -m "{msg}"
!git push origin main
print(f'Pushed: {msg}')